# Notebook 2 — SHAP: ครบ 10 เงื่อนไข

รันหลักบน Colab GPU และเก็บผลบน Google Drive อ่าน RUN_GUIDE.md ก่อนเริ่ม

Classification: Infiltration-only vs No Finding-only; balanced official test subset 100 ภาพ (ไม่ใช่ official test เต็ม)
ฝึก epoch ละ 320 ภาพรวม สุ่มจาก train pool แบบสมดุลคลาส ใช้ IDs เดียวกันทุกเงื่อนไขใน epoch เดียวกัน
XAI: bbox Infiltration 123 ภาพเดิม รวม mixed cases; คนไข้ไม่ซ้ำกับชุดฝึก

Baseline, Median L1–L3, CLAHE+DWT L1–L3, DAE+CLAHE L1–L3; ค่าทั้งหมดอยู่ใน shared/config.json

CLAHE+DWT เป็น joint presets (clip/depth/threshold); DAE+CLAHE เป็นการเปลี่ยน contrast หลัง DAE

## 1. สภาพแวดล้อมและเส้นทาง
อัปโหลด Notebook_2_colab.zip ไว้ที่ MyDrive/ แล้วเปิด notebook นี้ใน Colab เซลล์นี้แตกโค้ดลง Drive และข้อมูลที่เลือกแล้วลง runtime โดยอัตโนมัติ ไม่ต้องอัปโหลดภาพ NIH ทั้งหมด ใช้ OUTPUT_ROOT เดียวกันทั้งสาม notebook เลือก SMOKE ค่าเดียวกันเสมอ

In [ ]:
from pathlib import Path
import sys
import zipfile
import shutil

# Upload Notebook_2_colab.zip to My Drive; selected data is included.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")

PACKAGE_ZIP = Path("/content/drive/MyDrive/Notebook_2_colab.zip")
if IN_COLAB and PACKAGE_ZIP.is_file():
    # Keep source and results on Drive; extract selected image data to runtime disk below.
    NOTEBOOK_2_DIR = Path("/content/drive/MyDrive/Notebook_2")
    with zipfile.ZipFile(PACKAGE_ZIP) as package:
        for member in package.infolist():
            if member.is_dir() or member.filename == "Notebook_2/nih_cxr_subset.zip":
                continue
            relative = Path(member.filename)
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "Notebook_2":
                raise ValueError("Unsafe source package path")
            target = (NOTEBOOK_2_DIR.parent / relative).resolve()
            if not target.is_relative_to(NOTEBOOK_2_DIR.resolve()):
                raise ValueError("Source package leaves Notebook_2")
            target.parent.mkdir(parents=True, exist_ok=True)
            with package.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)
else:
    candidates = ([Path("/content/drive/MyDrive/Notebook_2")]
                  + [p.parent.parent for p in Path("/content/drive/MyDrive").glob("*/notebooks/Notebook_2/shared/config.json")]) if IN_COLAB else [
                      Path.cwd(), Path.cwd().parent, Path.cwd() / "Notebook_2", Path.cwd() / "notebooks/Notebook_2"]
    NOTEBOOK_2_DIR = next((p for p in candidates if (p / "shared/config.json").is_file()), None)
    if NOTEBOOK_2_DIR is None:
        raise FileNotFoundError("Upload Notebook_2_colab.zip to My Drive before running this cell")
# With the portable ZIP, DATA_ROOT is set to the extracted subset automatically.
DATA_ROOT = None
OUTPUT_ROOT = None  # Uses output_subdirectory in shared/config.json; same in all notebooks.
SMOKE = False  # True uses separate smoke_runs/ output and tiny validation data.

if IN_COLAB:
    # A previous failed cell may have imported shared/ from the old Drive folder.
    import importlib
    for module_name in tuple(sys.modules):
        if module_name == "shared" or module_name.startswith("shared."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "opencv-python-headless", "PyWavelets", "scikit-learn", "pandas", "matplotlib"])
sys.path.insert(0, str(NOTEBOOK_2_DIR))
if IN_COLAB:
    import os
    os.environ["CXR_PREPROCESS_CACHE_ROOT"] = "/content/nih_cxr_processed"
# Optional local test dependencies; nothing here changes the machine's global packages.
if not IN_COLAB and (NOTEBOOK_2_DIR / ".runtime").is_dir():
    sys.path.insert(0, str(NOTEBOOK_2_DIR / ".runtime"))

import torch
from shared.experiment import Experiment, versions
from shared.data import prepare_manifests
from shared.reporting import refresh_reports
if IN_COLAB and PACKAGE_ZIP.is_file() and DATA_ROOT is None:
    from shared.data_bundle import colab_data_from_package
    DATA_ROOT = colab_data_from_package(PACKAGE_ZIP, NOTEBOOK_2_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and DEVICE != "cuda":
    raise RuntimeError("Select Runtime > Change runtime type > GPU before running the experiment")
exp = Experiment.open(NOTEBOOK_2_DIR, DATA_ROOT, OUTPUT_ROOT, smoke=SMOKE)
print("Device:", DEVICE, "| Mode:", exp.config["run_mode"])
print("Data:", exp.data_root, "| Results:", exp.output_root)
print(versions())


## 2. ตรวจข้อมูลและใช้ manifests ร่วมกัน
สุ่ม train 160/คลาส และ validation 40/คลาสด้วย seed 42 แบบแยกคนไข้ ไม่มีการย้ายภาพระหว่าง official train_val/test

In [ ]:
manifests = prepare_manifests(exp)
for name, frame in manifests.items():
    print(name, "images:", len(frame), "patients:", frame.patient_id.nunique())
print("Patient/image leakage checks passed. Shared manifests are locked.")
display(manifests["xai"].risk_flag.value_counts().to_frame("n_images"))


## 3. SHAP ของ Infiltration logit
ใช้ shap.GradientExplainer (Expected Gradients) จริง ไม่ใช่ gradient×activation
Background มาจาก train คลาสละ 16 ภาพ ใช้ IDs เดียวกันทุกเงื่อนไข และ nsamples=200
เก็บ signed attribution ราย channel ที่ input resolution และ signed sum ที่ขนาดต้นฉบับ ก่อนแยกค่าบวก normalize เป็น [0,1] เพื่อประเมิน bbox
Overlay สีแดงเพิ่มคะแนน Infiltration สีน้ำเงินลดคะแนน Infiltration

In [ ]:
from shared.xai import run_xai
records = run_xai(exp, manifests, 'SHAP', DEVICE)
display(records.head())

## ผลเปรียบเทียบและ README
ตารางรวมและรูปเปรียบเทียบอยู่ใน comparison/ ของแต่ละเกณฑ์ CI ของ XAI ใช้ patient bootstrap 2,000 รอบ seed เดียวกัน รายงาน Overall/Pure/Mixed-high-risk/Mixed-low-risk; ผลที่ยังไม่รันระบุ not_run

In [ ]:
status = refresh_reports(exp)
display(status)
from IPython.display import Markdown, display
display(Markdown((exp.output_root / 'README.md').read_text(encoding='utf-8')))

## Export ผลลัพธ์ของรอบนี้
ดาวน์โหลด ZIP รวมผลและ checkpoint ที่มีอยู่ ไม่รวม cache ภาพ

In [ ]:
# Export this run's measured results and checkpoints; no retraining required.
from pathlib import Path
from datetime import datetime, timezone
import tempfile
import zipfile
import os

source = Path(exp.output_root).resolve()
if not source.is_dir():
    raise FileNotFoundError(source)
# Regenerate summaries before exporting the existing measured results.
refresh_reports(exp)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
export_dir = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
archive = export_dir / f"{source.name}_results_{stamp}.zip"
partial = archive.with_suffix(".zip.partial")
count = 0
try:
    with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1, allowZip64=True) as zipped:
        for path in sorted(source.rglob("*")):
            relative = path.relative_to(source)
            # Recomputable image caches are omitted; retain all result/provenance files.
            if any(part in {"cache", "__pycache__"} for part in relative.parts):
                continue
            if path.is_file():
                zipped.write(path, (Path(source.name) / relative).as_posix())
                count += 1
    if not count:
        raise RuntimeError("No result files found for export")
    os.replace(partial, archive)
except Exception:
    partial.unlink(missing_ok=True)
    raise
print(f"Exported {count} files: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Contains existing CNN / Grad-CAM / SHAP results, checkpoints and shared artifacts.")
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
    print("If the browser blocks the download, download this ZIP from Colab's Files panel.")
